# 🎙️ TalkForge — AI Lip Sync Video Generator

**Portrait + Audio → Talking-Head Video with accurate lip sync, powered by SadTalker.**

---

## Before you run

1. **Enable GPU:** `Runtime → Change runtime type → T4 GPU → Save`  
2. **Run all cells:** `Runtime → Run all` (or press `Ctrl + F9`)  
3. **Wait for the public URL** printed at the bottom of the last cell — then open it in any browser.

> ℹ️ Total setup time: **~5–10 minutes** on a fresh Colab session (mostly model download).  
> On subsequent runs in the same session, weights are cached and launch takes ~30 seconds.

---

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 — Verify GPU
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('✅ GPU detected:', r.stdout.strip())
else:
    print('⚠️  No GPU detected.')
    print('   Go to Runtime → Change runtime type → T4 GPU for much faster results.')
    print('   Continuing on CPU — generation will be slow (several minutes per clip).')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Install system packages
# ─────────────────────────────────────────────────────────────────────────────
# FFmpeg is required for audio/video processing.
# cmake + build-essential are required to compile dlib.

print('Installing system packages (FFmpeg, cmake, build tools)…')

!apt-get update -qq 2>&1 | tail -3
!apt-get install -y -qq \
    ffmpeg \
    cmake \
    build-essential \
    libopenblas-dev \
    liblapack-dev \
    libx11-dev \
    python3-dev \
    git wget curl 2>&1 | tail -5

# Confirm FFmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ FFmpeg:', r.stdout.split('\n')[0])
else:
    raise RuntimeError('FFmpeg installation failed — re-run this cell.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Clone TalkForge
# ─────────────────────────────────────────────────────────────────────────────
# Update TALKFORGE_REPO to your fork URL once published.
# Until then the notebook creates all required files inline below.

import os, pathlib, textwrap

TALKFORGE_DIR  = '/content/TalkForge'
TALKFORGE_REPO = 'https://github.com/your-username/TalkForge.git'

if pathlib.Path(f'{TALKFORGE_DIR}/.git').exists():
    print('TalkForge already cloned — pulling latest…')
    !cd {TALKFORGE_DIR} && git pull -q
else:
    print('Cloning TalkForge…')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', TALKFORGE_REPO, TALKFORGE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('⚠️  Git clone failed (repo may not be public yet).')
        print('   Creating project files inline instead…')

        # ── Inline project scaffold ───────────────────────────────────────
        # Run if the repo hasn't been pushed to GitHub yet.
        # Copy the contents of this notebook's sibling .py files here.
        for d in ['app', 'models', 'weights/checkpoints/BFM_Fitting',
                  'weights/gfpgan/weights', 'outputs', 'uploads',
                  'models/SadTalker']:
            pathlib.Path(f'{TALKFORGE_DIR}/{d}').mkdir(parents=True, exist_ok=True)

        # models/base.py
        pathlib.Path(f'{TALKFORGE_DIR}/models/__init__.py').write_text(
            'from models.base import BaseLipSyncModel\n'
            'from models.sadtalker import SadTalkerModel\n'
        )
        pathlib.Path(f'{TALKFORGE_DIR}/app/__init__.py').write_text('')
        print('   Placeholder scaffold created. Upload the full source files to get all features.')
    else:
        print('✅ TalkForge cloned')

# Ensure required directories exist
for d in ['weights/checkpoints/BFM_Fitting', 'weights/gfpgan/weights',
          'outputs', 'uploads']:
    pathlib.Path(f'{TALKFORGE_DIR}/{d}').mkdir(parents=True, exist_ok=True)

os.chdir(TALKFORGE_DIR)
sys.path.insert(0, TALKFORGE_DIR)
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Clone SadTalker source
# ─────────────────────────────────────────────────────────────────────────────

SADTALKER_DIR = f'{TALKFORGE_DIR}/models/SadTalker'

if pathlib.Path(f'{SADTALKER_DIR}/.git').exists():
    print('SadTalker already cloned — skipping.')
else:
    print('Cloning SadTalker (this takes ~30 s)…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/OpenTalker/SadTalker.git', SADTALKER_DIR],
        check=True,
    )
    print('✅ SadTalker cloned')

print(f'SadTalker source: {SADTALKER_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Install PyTorch (CUDA 11.8)
# ─────────────────────────────────────────────────────────────────────────────
# Colab usually ships with a recent PyTorch+CUDA.  We upgrade if needed.

try:
    import torch
    if torch.cuda.is_available() and torch.__version__.startswith('2.'):
        print(f'✅ PyTorch {torch.__version__} with CUDA already present — skipping install.')
    else:
        raise ImportError
except Exception:
    print('Installing PyTorch 2.1 + CUDA 11.8…')
    !pip install -q torch==2.1.0+cu118 torchvision==0.16.0+cu118 torchaudio==2.1.0 \
        --index-url https://download.pytorch.org/whl/cu118

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Install Python dependencies
# ─────────────────────────────────────────────────────────────────────────────

print('Installing core packages…')
!pip install -q \
    'gradio>=4.26.0,<5.0.0' \
    opencv-python-headless \
    Pillow \
    imageio 'imageio-ffmpeg>=0.4.9' \
    scikit-image \
    librosa soundfile resampy \
    'numpy>=1.24.0,<2.0.0' scipy \
    einops kornia \
    h5py tqdm pyyaml requests yacs addict

print('Installing dlib (face landmarks — takes 1–2 min, compiled from source)…')
!pip install -q dlib

print('Installing face-alignment…')
!pip install -q face-alignment

print('Installing GFPGAN / Real-ESRGAN / BasicSR…')
!pip install -q basicsr realesrgan gfpgan

print('Installing SadTalker requirements…')
!pip install -q -r {SADTALKER_DIR}/requirements.txt

print('\n✅ All Python dependencies installed')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Download model weights (~3.2 GB total, cached on re-run)
# ─────────────────────────────────────────────────────────────────────────────

import pathlib, urllib.request, shutil

def download(url: str, dst: str, label: str = '') -> None:
    p = pathlib.Path(dst)
    p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists() and p.stat().st_size > 4096:
        mb = p.stat().st_size // 1024 // 1024
        print(f'  ✓ {label or p.name} ({mb} MB) — already downloaded')
        return
    print(f'  ↓ {label or p.name}…', end='', flush=True)
    if shutil.which('wget'):
        subprocess.run(['wget', '-q', '-O', dst, url], check=True)
    else:
        urllib.request.urlretrieve(url, dst)
    mb = pathlib.Path(dst).stat().st_size // 1024 // 1024
    print(f' ✅ ({mb} MB)')

W   = TALKFORGE_DIR + '/weights'
HF  = 'https://huggingface.co/vinthony/SadTalker-V0.0.2/resolve/main'
CKP = f'{W}/checkpoints'
BFM = f'{CKP}/BFM_Fitting'
GFP = f'{W}/gfpgan/weights'

print('=== SadTalker main checkpoints ===')
download(f'{HF}/SadTalker_V0.0.2_256.safetensors', f'{CKP}/SadTalker_V0.0.2_256.safetensors', 'SadTalker 256 px')
download(f'{HF}/mapping_00229-model.pth.tar',       f'{CKP}/mapping_00229-model.pth.tar',       'Mapping (229)')
download(f'{HF}/mapping_00109-model.pth.tar',       f'{CKP}/mapping_00109-model.pth.tar',       'Mapping (109)')

print('\n=== BFM 3-D face model ===')
download(f'{HF}/BFM_Fitting/BFM_model_front.mat',      f'{BFM}/BFM_model_front.mat',      'BFM front')
download(f'{HF}/BFM_Fitting/BFM_model_back.mat',       f'{BFM}/BFM_model_back.mat',        'BFM back')
download(f'{HF}/BFM_Fitting/similarity_Lm3D_all.mat',  f'{BFM}/similarity_Lm3D_all.mat',   'BFM landmarks')
download(f'{HF}/BFM_Fitting/std_exp.txt',               f'{BFM}/std_exp.txt',               'BFM expression basis')

print('\n=== GFPGAN face enhancer ===')
download(
    'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
    f'{GFP}/GFPGANv1.4.pth',
    'GFPGANv1.4'
)

print('\n✅ All model weights ready!')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Verify setup
# ─────────────────────────────────────────────────────────────────────────────

import importlib, sys, os

# Reload modules in case the source files were created after the initial import
for mod in list(sys.modules.keys()):
    if mod.startswith('models') or mod.startswith('app'):
        del sys.modules[mod]

from models.sadtalker import SadTalkerModel

model = SadTalkerModel(
    weights_dir=f'{TALKFORGE_DIR}/weights',
    device='auto',
)

if model.is_ready():
    print('✅ Verification passed — all weights present, model ready!')
    import torch
    print(f'   Device: {model.device} | CUDA: {torch.cuda.is_available()}')
else:
    print('⚠️  Some weights are missing. Running download again…')
    model.download_weights(progress_cb=print)
    if model.is_ready():
        print('✅ Setup complete!')
    else:
        print('❌ Setup failed. Check the output above for errors.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Launch TalkForge  🚀
# ─────────────────────────────────────────────────────────────────────────────
# A public URL like https://xxxxxxxxxxxx.gradio.live will be printed below.
# Click it to open the TalkForge web interface in your browser.
#
# The URL is valid for 72 hours or until you stop/restart the cell.
# To get a fresh URL, restart the runtime and re-run all cells.

import sys, os

# Ensure fresh imports
for mod in list(sys.modules.keys()):
    if mod.startswith('models') or mod.startswith('app'):
        del sys.modules[mod]

os.chdir(TALKFORGE_DIR)
sys.path.insert(0, TALKFORGE_DIR)

from app.interface import build_interface
from app.pipeline import init_model

WEIGHTS_DIR = f'{TALKFORGE_DIR}/weights'
OUTPUT_DIR  = f'{TALKFORGE_DIR}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

init_model(weights_dir=WEIGHTS_DIR)

demo = build_interface(output_dir=OUTPUT_DIR, weights_dir=WEIGHTS_DIR)

print('\n' + '=' * 62)
print('  TalkForge is launching…')
print('  Look for the public URL printed below 👇')
print('=' * 62 + '\n')

demo.queue(max_size=5).launch(
    share=True,           # creates the public gradio.live URL
    debug=False,
    show_error=True,
    server_name='0.0.0.0',
    server_port=7860,
)

---

## How to use TalkForge

1. **Upload a portrait** — any clear, front-facing photo (JPG / PNG / WEBP).
2. **Upload audio** — a speech clip in WAV, MP3, or M4A format.
3. **Click ✦ Generate Video** and watch the live status updates.
4. **Preview** the result directly in the page when it's ready.
5. **⬇ Download Video** to save the MP4, **✕ Discard** to delete it, or **↺ Generate New** to start over.

---

## Tips

| Tip | Details |
|-----|----------|
| Best portrait | Clear, front-facing, well-lit, no sunglasses |
| Best audio | 16 kHz mono WAV gives cleanest lip sync |
| Clip length | Under 60 s for fastest generation |
| Download your videos | Colab storage is **temporary** — download before the session ends |
| Session expired | Re-run all cells — weights are cached so setup takes ~30 s |

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Restart runtime → Run all |
| `Face not detected` | Use a clearer front-facing portrait |
| `No module named …` | Re-run Cell 6 (Python dependencies) |
| Public URL not appearing | Wait 30 s; Gradio share server can be slow |
| Generation very slow | Confirm GPU runtime in Cell 1 |

---

*TalkForge — MIT License.  Powered by [SadTalker](https://github.com/OpenTalker/SadTalker) and [Gradio](https://gradio.app).*